# AIML-02 training on Kaggle (free GPU)

1. Kaggle → *Create* → *New Notebook* → *File* → *Import Notebook* → this file.
2. *Add Input* → competition **rsna-pneumonia-detection-challenge** (accept the rules first).
3. *Settings* → Accelerator **GPU T4 x2** (or P100), Internet **On**.
4. *Add-ons* → *Secrets*: `WANDB_API_KEY` (optional but recommended), `GITHUB_REPO` = your repo URL.
5. *Run All*. Download `models/` and `reports/` from the Output tab when it finishes.

**Or from the laptop, no GitHub needed:** `python scripts/kaggle_push.py` uploads the source as a private dataset and starts this notebook on a GPU (`--status`, `--fetch` to follow up).

Everything here runs the **same DVC pipeline** as your laptop, so results are reproducible anywhere.

In [ ]:
# Run mode, set by scripts/kaggle_push.py: full (train + evaluate both models), eval_only (re-evaluate the released
# classifier, no training), seg_cv (5-fold cross-validation of the spleen model only)
import os
os.environ['AIML02_MODE'] = 'full'

In [ ]:
import glob, os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return None
if secret('WANDB_API_KEY'):
    os.environ['WANDB_API_KEY'] = secret('WANDB_API_KEY')
else:
    print('no W&B key: training will still run, just without live dashboards')
# Source: GitHub if the GITHUB_REPO secret is set, else the private dataset made by scripts/kaggle_push.py
repo = secret('GITHUB_REPO')
if repo:
    !git clone --depth 1 {repo} /kaggle/working/repo
else:
    # Kaggle unpacks archives on dataset upload, so look for either the tarball or the unpacked tree
    tarballs = glob.glob('/kaggle/input/**/repo.tar.gz', recursive=True)
    if tarballs:
        !mkdir -p /kaggle/working/repo && tar xzf {tarballs[0]} -C /kaggle/working/repo
    else:
        src_root = os.path.dirname(next(p for p in glob.glob('/kaggle/input/**/pyproject.toml', recursive=True)
                                        if os.path.isdir(os.path.join(os.path.dirname(p), 'src', 'medimg'))))
        !cp -r {src_root} /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q 'monai>=1.4' nibabel 'pydicom>=3.0' 'dvc>=3.55' wandb && pip install -q --no-deps -e .
!dvc config core.no_scm true  # the dataset copy has no .git
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Point the pipeline at the read-only Kaggle input instead of copying 4 GB.
rsna = os.path.dirname(glob.glob('/kaggle/input/**/stage_2_train_labels.csv', recursive=True)[0])
!mkdir -p data/raw && ln -sfn {rsna} data/raw/rsna
!ls data/raw/rsna | head

In [ ]:
%%bash
# Classification: prepare -> train -> evaluate (about 1 h on a T4). The stages run directly instead of via
# `dvc repro`: DVC would hash 4 GB of read-only DICOMs and copy every processed PNG into its cache.
# eval_only (`kaggle_push.py --eval-only`): the released classifier.pt is attached and training is skipped.
set -euo pipefail
export TQDM_DISABLE=1
if [ "$AIML02_MODE" = seg_cv ]; then echo 'seg_cv: skipping classification'; exit 0; fi
python -m medimg.data.prepare_rsna
if [ "$AIML02_MODE" = eval_only ]; then
  CKPT=$(find /kaggle/input -name classifier.pt -print -quit)
  echo "eval-only: evaluating the released checkpoint $CKPT"; mkdir -p models && cp "$CKPT" models/
else
  python -m medimg.train_cls
fi
python -m medimg.evaluate_cls
if [ "$AIML02_MODE" = eval_only ]; then
  # External validation on CheXpert (attached by kaggle_push.py --eval-only) and the demo pack of held-out test DICOMs
  CX=$(dirname "$(find /kaggle/input -maxdepth 5 -name valid.csv -print -quit)")
  echo "CheXpert at $CX"
  python -m medimg.external_cls --root "$CX"
  python scripts/export_demo_pack.py --n 24 --out demo_pack
fi

In [ ]:
import json, pandas as pd
if os.environ['AIML02_MODE'] != 'seg_cv':
    print(json.dumps(json.load(open('reports/metrics.json')), indent=2))
    display(pd.read_csv('reports/subgroups.csv'))

In [ ]:
%%bash
# 3D segmentation (spleen CT, downloads 1.5 GB from the MSD mirror).
#   full:   one 33/8 split, 600 epochs, about 2-2.5 h on a T4
#   seg_cv: 5-fold cross-validation over all 41 labelled CTs, 300 epochs per fold, no checkpoint selection (~4-5 h)
set -euo pipefail
export TQDM_DISABLE=1
case "$AIML02_MODE" in
  eval_only) echo 'eval_only: skipping segmentation' ;;
  seg_cv)    python -u -m medimg.seg.cv_seg ;;
  *)         python -u -m medimg.seg.train_seg && python -m medimg.seg.figures_seg ;;
esac

In [ ]:
# Bundle what the laptop needs, then drop the working copy so the run's output is just this archive
# (otherwise Kaggle lists tens of thousands of processed PNGs and downloading the outputs takes hours).
%cd /kaggle/working
!tar czf aiml02_outputs.tgz -C repo models reports $(cd repo && ls -d demo_pack 2>/dev/null)
!rm -rf repo && ls -lh /kaggle/working